In [ ]:
# ============================================================
# Cell W2-1: Detect language of each review
# ============================================================
from langdetect import detect, DetectorFactory
import pandas as pd

DetectorFactory.seed = 42

df = pd.read_csv('/content/data/airasia_raw.csv')

def safe_detect(text):
    try:
        return detect(str(text))
    except:
        return 'unknown'

print("🔍 Detecting languages... (1-2 mins)")
df['lang'] = df['review_text'].apply(safe_detect)

print("\n=== Language Distribution ===")
print(df['lang'].value_counts().head(10))
print(f"\nUnique languages: {df['lang'].nunique()}")

In [ ]:
# ============================================================
# Cell W2-2: Filter to English reviews only
# ============================================================

import os
import pandas as pd

before      = len(df)
df_excluded = df[df['lang'] != 'en'].copy()
df          = df[df['lang'] == 'en'].copy()
after       = len(df)

print(f"Before  : {before:,}")
print(f"Kept    : {after:,} English reviews")
print(f"Excluded: {before - after:,} non-English reviews")

print(f"\nExcluded language breakdown:")
print(df_excluded['lang'].value_counts().head(10))

os.makedirs('/content/data', exist_ok=True)
df_excluded.to_csv('/content/data/excluded_non_english.csv', index=False)
print("\n✅ Excluded reviews saved.")

In [ ]:
# ============================================================
# Cell W2-3: Remove emoji-only entries and spam/noise
# ============================================================
import re

# Define emoji_pattern here — also used in Cell W2-4
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "]+", flags=re.UNICODE
)

def is_emoji_only(text):
    stripped = emoji_pattern.sub('', str(text))
    stripped = re.sub(r'[\s\W]+', '', stripped)
    return len(stripped) == 0

before      = len(df)
df['is_emoji_only'] = df['review_text'].apply(is_emoji_only)
emoji_count = df['is_emoji_only'].sum()
df          = df[~df['is_emoji_only']].copy()
df.drop(columns=['is_emoji_only'], inplace=True)
print(f"Removed {emoji_count} emoji-only rows.")

def has_non_latin_script(text):
    cjk = re.findall(
        r'[\u4e00-\u9fff\u3040-\u30ff\u0e00-\u0e7f]', str(text))
    return len(cjk) > len(str(text)) * 0.3

before2    = len(df)
spam_count = df['review_text'].apply(has_non_latin_script).sum()
df         = df[~df['review_text'].apply(has_non_latin_script)].copy()
print(f"Removed {spam_count} non-Latin script spam rows.")
print(f"\n✅ Dataset after noise removal: {len(df):,} reviews")

In [ ]:
# ============================================================
# Cell W2-4: Core text cleaning
# ============================================================
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = emoji_pattern.sub('', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['review_text'].apply(clean_text)

before = len(df)
df     = df[df['cleaned_text'].str.strip() != ''].copy()
print(f"Removed {before - len(df)} rows that became empty after cleaning.")

print("\nSample before/after:")
for i in range(3):
    print(f"\nOriginal : {df['review_text'].iloc[i][:100]}")
    print(f"Cleaned  : {df['cleaned_text'].iloc[i][:100]}")

In [ ]:
# ============================================================
# Cell W2-5: Tokenize, remove stopwords, lemmatize
# ============================================================
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
negations  = {'no', 'not', 'nor', "n't", 'never', 'none'}
stop_words = stop_words - negations
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(str(text))
    tokens = [t for t in tokens
              if t not in stop_words and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

print("🔄 Tokenizing and lemmatizing... (1-2 mins)")
df['tokens']         = df['cleaned_text'].apply(preprocess_text)
df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))

before = len(df)
df     = df[df['tokens'].apply(len) > 0].copy()
print(f"Removed {before - len(df)} empty token rows.")

print("\nSample:")
for i in range(2):
    print(f"\nCleaned   : {df['cleaned_text'].iloc[i]}")
    print(f"Processed : {df['processed_text'].iloc[i]}")

In [ ]:
# ============================================================
# Cell W2-6: Word clouds per sentiment class
# ============================================================
from wordcloud import WordCloud
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, label, cmap in zip(
        axes,
        ['positive', 'neutral', 'negative'],
        ['Greens',   'Oranges', 'Reds']):

    blob = ' '.join(df[df['sentiment'] == label]['processed_text'])
    wc   = WordCloud(width=600, height=400,
                     background_color='white',
                     colormap=cmap,
                     max_words=80).generate(blob)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{label.capitalize()}', fontsize=14)
    ax.axis('off')

plt.suptitle('AirAsia Reviews — Word Clouds by Sentiment', fontsize=16)
plt.tight_layout()
plt.savefig('/content/data/wordclouds.png', dpi=150)
plt.show()
print("✅ Word clouds saved.")

In [ ]:
# ============================================================
# Cell W2-7: Export final cleaned dataset
# ============================================================
import pandas as pd
import os

final_df = df[[
    'reviewId', 'userName', 'review_date',
    'review_text', 'cleaned_text', 'processed_text',
    'tokens', 'star_rating', 'sentiment', 'lang',
    'thumbsUpCount'
]].copy()

# Convert token lists to string for CSV compatibility
final_df['tokens'] = final_df['tokens'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else str(x))

os.makedirs('/content/data', exist_ok=True)
final_df.to_csv('/content/data/cleaned_data.csv', index=False)

raw_count = pd.read_csv('/content/data/airasia_raw.csv').shape[0]

print("=" * 50)
print("  Week 2 Preprocessing — Summary")
print("=" * 50)
print(f"Original reviews : {raw_count:,}")
print(f"Final cleaned    : {len(final_df):,}")
print(f"Removed total    : {raw_count - len(final_df):,}")
print(f"\nFinal sentiment distribution:")
print(final_df['sentiment'].value_counts())
print(f"\nPercentage:")
print(final_df['sentiment'].value_counts(normalize=True)
      .mul(100).round(1).astype(str) + '%')
print("\n✅ Saved to /content/data/cleaned_data.csv")

In [ ]:
# ============================================================
# Cell W2-8: Train / Validation / Test split (70/20/10)
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/data/cleaned_data.csv')
df = df.dropna(subset=['processed_text'])
df = df[df['processed_text'].str.strip() != '']

X = df['processed_text']
y = df['sentiment']

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

# 20% test, 10% val from the 30% temp
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=1/3, random_state=42, stratify=y_temp)

print(f"Total    : {len(df):,}")
print(f"Train    : {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test     : {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)")
print(f"Val      : {len(X_val):,} ({len(X_val)/len(df)*100:.1f}%)")

print(f"\nClass balance (train):")
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# Cell W2-9: Oversample minority classes (training set only)
# ============================================================
from imblearn.over_sampling import RandomOverSampler
import numpy as np

print("Before oversampling:")
print(y_train.value_counts())

ros = RandomOverSampler(random_state=42)
X_train_arr = X_train.values.reshape(-1, 1)
X_train_resampled, y_train_resampled = ros.fit_resample(
    X_train_arr, y_train)
X_train_resampled = X_train_resampled.flatten()

print("\nAfter oversampling:")
print(pd.Series(y_train_resampled).value_counts())
print("\n✅ Oversampling done — val/test sets untouched.")

In [ ]:
# ============================================================
# Cell W2-10: Train Naive Bayes with TF-IDF
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import time

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_resampled)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")

start    = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train_resampled)
nb_train_time = time.time() - start

print(f"✅ Naive Bayes trained in {nb_train_time:.2f} seconds")

In [ ]:
# ============================================================
# Cell W2-11: Evaluate Naive Bayes
# ============================================================
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model(model, X, y_true, name="Model", split="Test"):
    y_pred = model.predict(X)
    acc    = accuracy_score(y_true, y_pred)
    prec   = precision_score(y_true, y_pred,
                              average='weighted', zero_division=0)
    rec    = recall_score(y_true, y_pred,
                           average='weighted', zero_division=0)
    f1     = f1_score(y_true, y_pred,
                       average='weighted', zero_division=0)

    print(f"=== {name} — {split} ===")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"\n{classification_report(y_true, y_pred, zero_division=0)}")

    labels = sorted(y_true.unique())
    cm     = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'{name} — Confusion Matrix ({split})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(
        f'/content/data/nb_confusion_{split.lower()}.png', dpi=150)
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

nb_val_results  = evaluate_model(
    nb_model, X_val_tfidf,  y_val,  "Naive Bayes", "Validation")
nb_test_results = evaluate_model(
    nb_model, X_test_tfidf, y_test, "Naive Bayes", "Test")